# MedSegDiff Thigh Segmentation — Augmented Dataset (Lambda)

Runs **inference only** using already-trained MedSegDiff checkpoints on the
20 augmented NIfTI water volumes.

The model was trained with two channels: water (ch0) + fat-fraction (ch1).
If a fat-fraction file does not exist for a given stem, the fat channel is zeros.

⚠️ **Checkpoints must exist** in `~/medsegdiff_ckpts/` (trained on myosegmenTUM).

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`
Fat (optional): `~/our_augmented_dataset/{stem}_augmented000_fat.nii.gz`
Output: `~/medsegdiff_augmented_segs/{stem}_medsegdiff.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/medsegdiff_ckpts/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_ckpts/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/medsegdiff/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_augmented_segs/ \
  /path/to/local/medsegdiff/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def _ensure(*pkgs):
    import importlib
    missing = [p for p in pkgs
               if importlib.util.find_spec(p.replace('-', '_')) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(missing))
    else:
        print('Already installed:', ', '.join(pkgs))

_ensure('SimpleITK', 'tqdm', 'torchvision')

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import glob, os, sys
import numpy as np
import torch
import torch.nn.functional as F
import SimpleITK as sitk

MEDSEGDIFF_DIR = os.path.expanduser('~/medsegdiff')
CKPT_DIR       = os.path.expanduser('~/medsegdiff_ckpts')
DATA_DIR       = os.path.expanduser('~/our_augmented_dataset')
OUTPUT_DIR     = os.path.expanduser('~/medsegdiff_augmented_segs')

for path, label in [
    (MEDSEGDIFF_DIR, 'medsegdiff package'),
    (CKPT_DIR,       'checkpoints'),
    (DATA_DIR,       'augmented dataset'),
]:
    ok = os.path.isdir(path)
    print(f'{"OK" if ok else "MISSING"}: {label}')
    if not ok:
        raise FileNotFoundError(f'Upload {label} first')

os.makedirs(OUTPUT_DIR, exist_ok=True)

if MEDSEGDIFF_DIR not in sys.path:
    sys.path.insert(0, MEDSEGDIFF_DIR)

from dataset   import GT_LABELS, _norm
from unet      import UNet
from diffusion import GaussianDiffusion

IMG_SIZE   = 256
BASE_CH    = 64
T_DIM      = 256
T_STEPS    = 1000
DDIM_STEPS = 50
USE_FF     = True   # 2-channel: water ch0, fat ch1; if fat absent use zeros
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MUSCLES    = list(GT_LABELS)

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'Muscles : {MUSCLES}')
print(f'Volumes : {len(nii_files)}')
print(f'Device  : {DEVICE}')

In [ ]:
models    = {}
diffusion = GaussianDiffusion(T=T_STEPS, device=DEVICE)

for muscle in MUSCLES:
    best_ckpt = os.path.join(CKPT_DIR, f'{muscle}_best.pt')
    if not os.path.exists(best_ckpt):
        print(f'[WARN] checkpoint not found: {best_ckpt}')
        continue
    ckpt       = torch.load(best_ckpt, map_location=DEVICE)
    saved_args = ckpt.get('args', {})
    img_ch     = saved_args.get('img_ch', 2)
    model = UNet(
        img_ch=img_ch,
        base=saved_args.get('base_ch', BASE_CH),
        t_dim=saved_args.get('t_dim', T_DIM),
    ).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    models[muscle] = (model, img_ch)
    print(f'  {muscle}: epoch {ckpt.get("epoch","?")}, '
          f'best Dice {ckpt.get("best_dice",0):.4f}, img_ch={img_ch}')

print(f'Loaded {len(models)}/{len(MUSCLES)} models.')

In [ ]:
@torch.no_grad()
def segment_stack(model, img_ch, w_arr, ff_arr):
    """Segment a volume slice-by-slice. water=ch0, fat=ch1 (zeros if absent)."""
    D, H, W  = w_arr.shape
    pred_vol = np.zeros((D, H, W), dtype=np.uint8)
    model.eval()
    for sl in range(D):
        channels = [torch.from_numpy(_norm(w_arr[sl])).unsqueeze(0)]
        if img_ch == 2:
            if ff_arr is not None:
                channels.append(torch.from_numpy(_norm(ff_arr[sl])).unsqueeze(0))
            else:
                channels.append(torch.zeros_like(channels[0]))
        img_t = torch.cat(channels, dim=0) * 2.0 - 1.0
        img_r = F.interpolate(
            img_t.unsqueeze(0).to(DEVICE),
            size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False,
        )
        pred  = diffusion.ddim_sample(model, img_r, num_steps=DDIM_STEPS)
        pred_r= F.interpolate(pred, size=(H, W), mode='bilinear', align_corners=False)
        pred_vol[sl] = (pred_r.squeeze().cpu().numpy() > 0).astype(np.uint8)
    return pred_vol


for nii_path in nii_files:
    basename = os.path.basename(nii_path)
    stem     = basename.replace('_water.nii.gz', '')
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_medsegdiff.npz')

    if os.path.exists(out_path):
        existing = set(np.load(out_path).files)
        if set(models.keys()).issubset(existing):
            print(f'Skipping (done): {stem}')
            continue

    print(f'\nProcessing: {stem}')
    w_arr  = sitk.GetArrayFromImage(sitk.ReadImage(nii_path)).astype(np.float32)
    D, H, W = w_arr.shape

    # fat-fraction: replace _water.nii.gz with _fat.nii.gz
    fat_path = nii_path.replace('_water.nii.gz', '_fat.nii.gz')
    ff_arr = sitk.GetArrayFromImage(sitk.ReadImage(fat_path)).astype(np.float32) \
             if os.path.exists(fat_path) else None
    print(f'  Shape: {w_arr.shape}  fat available: {ff_arr is not None}')

    all_masks = {}
    if os.path.exists(out_path):
        all_masks = dict(np.load(out_path))

    for muscle, (model, img_ch) in models.items():
        if muscle in all_masks:
            print(f'  {muscle}: already done')
            continue
        print(f'  {muscle} ...', end=' ', flush=True)
        pred = segment_stack(model, img_ch, w_arr, ff_arr)
        all_masks[muscle] = pred
        print(f'{int(pred.sum()):,} voxels')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
# Sanity check
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    s = np.load(results[0])
    print(f'Sample: {results[0]}')
    for k in sorted(s.files):
        print(f'  {k}: shape={s[k].shape}  voxels={int(s[k].sum()):,}')